# RUNG-26c specificity — AF2 cross-check of the PXDesign rank_1 binder

Independent of the Protenix webserver: fold the PXDesign passing designs (rank_1..3) vs the **MUT** pMHC (IDH1-R132H) AND the **WT** pMHC on AF2/ColabDesign (same scorer family as the design's AF2-IG number), compute discrimination = pae_interaction(WT) - pae_interaction(MUT). mutant_specific = binder on MUT (pae<=10, plddt>=80) AND discrimination >= 3. Reuses the RUNG-26-proven score() + Drive meta.json (mut_pdb/wt_pdb/hotspot). T4-OK. Cell1 install -> Cell2 score.

In [ ]:
#@title Cell 1 — clone + GPU-guard + install ColabDesign (jax; NO torch)  [~5 min]
import os, glob, subprocess
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!nvidia-smi -L
from google.colab import drive; drive.mount('/content/drive')
def sh(c): r=subprocess.run(c,shell=True,capture_output=True,text=True); (r.returncode and print('  !',(r.stderr or '')[-300:])); return r.returncode
sh('pip install -q gemmi git+https://github.com/sokrypton/ColabDesign.git@v1.1.1')
if not glob.glob('params/params_model_*.npz'):
    sh('apt-get -qq install -y aria2 >/dev/null; mkdir -p params && cd params && (aria2c -q -x16 https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar || wget -q https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar) && tar -xf alphafold_params_2022-12-06.tar && rm -f alphafold_params_2022-12-06.tar')
import jax
assert any(d.platform=='gpu' for d in jax.devices()), 'NO GPU — Runtime > Change runtime type > T4 GPU, then rerun Cell 1'
print('jax GPU:', jax.devices())
from colabdesign import mk_afdesign_model; print('colabdesign import OK')
print('[CELL 1] done')


In [ ]:
#@title Cell 2 — score PXDesign rank_1..3 vs MUT & WT pMHC -> discrimination
import os, csv, json
from google.colab import drive; drive.mount('/content/drive')
from colabdesign import mk_afdesign_model, clear_mem
import jax
assert any(d.platform=='gpu' for d in jax.devices()), 'NO GPU — switch to T4 and rerun'
M=json.load(open('/content/drive/MyDrive/cancer-recon/rung26b_rfdiff/meta.json'))
MUT_PDB=M['mut_pdb']; WT_PDB=M['wt_pdb']; HOT=f"B{M['hotspot']}"
print('MUT_PDB exists:', os.path.exists(MUT_PDB), MUT_PDB)
print('WT_PDB  exists:', os.path.exists(WT_PDB), WT_PDB)
def score(pdb,seq):
    clear_mem(); m=mk_afdesign_model(protocol='binder',use_multimer=False,num_recycles=3,data_dir='.')
    m.prep_inputs(pdb_filename=pdb,chain='A,B',binder_len=len(seq),hotspot=HOT,rm_target_seq=False,ignore_missing=True)
    m.predict(seq=seq,verbose=False); log=m.aux['log']
    return {'pae_interaction':round(float(log['i_pae'])*31.0,3),'binder_plddt':round(float(log['plddt'])*100.0,1)}
SUM='runs/rung26c_pxdesign_idh1/design_outputs/protenix_design_989b6522/summary.csv'
rows=list(csv.DictReader(open(SUM)))
PASS=[r for r in rows if r.get('AF2-IG-easy-success')=='True'] or rows[:3]
print(f'scoring {len(PASS)} PXDesign designs vs MUT+WT (independent AF2 cross-check of the webserver)')
results=[]
for r in PASS:
    rk=r['rank']; seq=r['sequence'].strip()
    mut=score(MUT_PDB,seq)
    wt=score(WT_PDB,seq) if os.path.exists(WT_PDB) else None
    disc=round(wt['pae_interaction']-mut['pae_interaction'],3) if wt else None
    is_binder = mut['pae_interaction']<=10.0 and mut['binder_plddt']>=80.0
    mut_specific = bool(is_binder and disc is not None and disc>=3.0)
    results.append({'rank':rk,'len':len(seq),'mut':mut,'wt':wt,'discrimination_pae_wt_minus_mut':disc,
                    'is_binder':is_binder,'mutant_specific':mut_specific,
                    'pxdesign_af2ig_iptm':r.get('af2_iptm'),'pxdesign_af2ig_ipAE':r.get('af2_ipAE')})
    print(f"  rank{rk}: MUT pae={mut['pae_interaction']} plddt={mut['binder_plddt']} | WT pae={wt['pae_interaction'] if wt else 'NA'} | DISC(wt-mut)={disc} | binder={is_binder} specific={mut_specific}", flush=True)
out={'tag':'rung26c_pxdesign_specificity_af2','target':'IDH1_R132H_A0101','hotspot':HOT,
     'thresholds':{'pae_bind':10.0,'plddt_min':80.0,'discrimination_min':3.0},'designs':results}
os.makedirs('runs/rung26c_pxdesign_idh1',exist_ok=True)
json.dump(out, open('runs/rung26c_pxdesign_idh1/specificity_af2.json','w'), indent=1)
print('\nVERDICT:', 'MUTANT-SPECIFIC binder FOUND (AF2)' if any(d['mutant_specific'] for d in results) else 'none mutant-specific by AF2 — read DISC values (WT-MUT; >=3 = specific)')
try:
    from google.colab import files; files.download('runs/rung26c_pxdesign_idh1/specificity_af2.json')
except Exception as e: print('(skip dl:',e,')')
